# HGB Compact Ladder: 104 → 80 → 60 → 40

2019–2022 학습 / 2023 선택으로 피처셋을 확정한 뒤, 2019–2023 재학습 / 2024 최종 비교를 수행합니다.
HGB는 모든 단계에서 max_iter=350으로 고정합니다.


In [ ]:
from pathlib import Path
if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    from google.colab import drive
    MOUNT_POINT = "/content/gdrive_hgb_v2"
    drive.mount(MOUNT_POINT)
    DRIVE_ROOT = Path(MOUNT_POINT) / "MyDrive"
assert DRIVE_ROOT.is_dir()
print(DRIVE_ROOT)


In [ ]:
import shutil, subprocess
REPO_DIR = Path("/content/lg_aimers_experiment_lab_v2")
BRANCH = "agent/hgb-feature-selection-v2"
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git","clone","--depth","1","--branch",BRANCH,"https://github.com/tswaincae1221/lg_aimers_experiment_lab.git",str(REPO_DIR)], check=True)
subprocess.run(["python","-m","pip","install","-q","-r",str(REPO_DIR / "requirements.txt")], check=True)
print("Repository ready:", REPO_DIR)


In [ ]:
def resolve_drive_file(filename: str, preferred_folder: str = "aimers_data") -> Path:
    preferred = DRIVE_ROOT / preferred_folder / filename
    if preferred.is_file(): return preferred
    matches = sorted(p for p in DRIVE_ROOT.rglob(filename) if p.is_file())
    if len(matches) == 1: return matches[0]
    raise RuntimeError(f"Could not uniquely resolve {filename}: {matches[:20]}")
TRAIN_PATH = resolve_drive_file("train.csv")
TRACKMAN_PATH = resolve_drive_file("trackman_history.csv")
MAPPING_PATH = REPO_DIR / "resources" / "pitcher_trackman_mapping.csv"
FULL_RESULTS_DIR = DRIVE_ROOT / "aimers_data" / "results" / "hgb_feature_selection_v2_full"
OUTPUT_DIR = DRIVE_ROOT / "aimers_data" / "results" / "hgb_compact_ladder_104_80_60_40"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("TRAIN", TRAIN_PATH)
print("TRACKMAN", TRACKMAN_PATH)
print("CACHE", FULL_RESULTS_DIR / "cache")
print("OUTPUT", OUTPUT_DIR)


In [ ]:
import subprocess, sys
cmd = [
    sys.executable, "-m", "src.hgb_compact_ladder_experiment",
    "--train", str(TRAIN_PATH),
    "--trackman", str(TRACKMAN_PATH),
    "--mapping", str(MAPPING_PATH),
    "--output-dir", str(OUTPUT_DIR),
    "--trackman-cache-dir", str(FULL_RESULTS_DIR),
    "--selection-season", "2023",
    "--validation-season", "2024",
    "--selection-permutation-sample", "50000",
    "--selection-permutation-repeats", "3",
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


In [ ]:
import json, pandas as pd
print("=== Final 2024 comparison ===")
display(pd.read_csv(OUTPUT_DIR / "compact_model_scores.csv"))
print("=== Selection stages ===")
display(pd.read_csv(OUTPUT_DIR / "selection_stage_summary.csv"))
print("=== Summary ===")
print(json.dumps(json.load(open(OUTPUT_DIR / "compact_ladder_summary.json", encoding="utf-8")), ensure_ascii=False, indent=2))


## Selection rule

- Candidate pool: Full 118 minus the 14-feature `recent_form` block = 104.
- Feature ranking is never computed on 2024. 2019–2022 train and 2023 validation are used for selection.
- At each pruning stage, Brier permutation importance is recalculated.
- Ranking uses `robust_importance = mean(delta Brier) - std(delta Brier)`.
- Remove the lowest robust-importance features first.
- Keep at least one feature from every remaining semantic block.
- Always protect `season` and `season_trend_prior`, because within a one-season holdout they are constant and ordinary permutation importance cannot assess them fairly.
- The resulting 104/80/60/40 sets are finally retrained on 2019–2023 and evaluated once on 2024.
